# ATP Tennis Match Predictor

This notebook walks through the full pipeline:

1. **Data loading** — historical ATP match records (1991–2025)
2. **Feature engineering** — ELO ratings, head-to-head records, rolling serve statistics
3. **Model training** — Random Forest classifier
4. **Prediction** — win probabilities for any two players on any surface

All logic is imported from `predict.py`, so this notebook serves as an
interactive companion rather than duplicating code.

## 1  Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from predict import (
    load_match_data,
    load_players,
    load_rankings,
    find_player,
    build_features_and_state,
    train_model,
    build_prediction_vector,
    WIN_RATE_WINDOWS,
    SERVE_STAT_WINDOWS,
    ELO_GRAD_WINDOWS,
    SERVE_STAT_PREFIX,
    ELO_DEFAULT,
)
from collections import deque

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## 2  Load raw data

In [ ]:
players_df = load_players()
rankings   = load_rankings()
matches_df = load_match_data()

print(f"Players loaded:  {len(players_df):,}")
print(f"Rankings loaded: {len(rankings):,}")
print(f"Matches loaded:  {len(matches_df):,}")
matches_df.head(3)

### 2.1  Surface distribution

In [ ]:
surface_counts = matches_df["surface"].value_counts()

fig, ax = plt.subplots(figsize=(6, 3))
surface_counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.set_title("Matches by surface")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.2  Matches per year

In [ ]:
matches_per_year = (
    matches_df["tourney_date"]
    .astype(str)
    .str[:4]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 3))
matches_per_year.plot(kind="bar", ax=ax, color="#4C72B0", width=0.85)
ax.set_title("Matches per year in dataset")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3  Feature engineering

This step builds all features from scratch by replaying the match history
in chronological order.  It also returns `player_state` — a snapshot of
every player's current ELO, head-to-head record, and rolling statistics
after the last match in the dataset.

In [ ]:
feature_df, player_state = build_features_and_state(matches_df)

print(f"Feature matrix shape: {feature_df.shape}")
print(f"Features ({len(feature_df.columns) - 1}):")
for col in feature_df.columns[:-1]:
    print(f"  {col}")

### 3.1  Feature correlation heatmap (subset)

In [ ]:
# Show a readable subset of the most interpretable features
subset_cols = [
    "ATP_POINT_DIFF", "ATP_RANK_DIFF", "AGE_DIFF",
    "H2H_DIFF", "H2H_SURFACE_DIFF",
    "WIN_LAST_10_DIFF", "WIN_LAST_50_DIFF",
    "ELO_DIFF", "ELO_SURFACE_DIFF",
    "RESULT",
]

corr = feature_df[subset_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(subset_cols)))
ax.set_yticks(range(len(subset_cols)))
ax.set_xticklabels(subset_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(subset_cols, fontsize=8)
ax.set_title("Feature correlation (subset)")
plt.tight_layout()
plt.show()

## 4  Model training

In [ ]:
model, feature_names = train_model(feature_df)

### 4.1  Feature importances

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_names)
top_n = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
top_n.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Top 20 feature importances (Random Forest)")
ax.set_xlabel("Mean impurity decrease")
plt.tight_layout()
plt.show()

## 5  Making a prediction

Edit the cell below to try any two players and surface.

In [ ]:
PLAYER_1 = "Novak Djokovic"
PLAYER_2 = "Carlos Alcaraz"
SURFACE  = "clay"   # "hard", "clay", or "grass"

In [ ]:
p1_id, p1_name = find_player(PLAYER_1, players_df)
p2_id, p2_name = find_player(PLAYER_2, players_df)
print(f"Resolved: '{PLAYER_1}' -> {p1_name} (id={p1_id})")
print(f"Resolved: '{PLAYER_2}' -> {p2_name} (id={p2_id})")

In [ ]:
x = build_prediction_vector(
    p1_id, p2_id, SURFACE,
    player_state, players_df, rankings, feature_names,
)

proba   = model.predict_proba(x.reshape(1, -1))[0]
classes = list(model.classes_)
p1_prob = proba[classes.index("Player 1 Wins")]
p2_prob = 1 - p1_prob

winner     = p1_name if p1_prob >= 0.5 else p2_name
confidence = max(p1_prob, p2_prob)

print("=" * 50)
print(f"  {p1_name} vs {p2_name}  ({SURFACE.capitalize()})")
print(f"  Expected winner: {winner}")
print(f"  Confidence:      {confidence:.1%}")
print(f"  {p1_name}: {p1_prob:.1%}  |  {p2_name}: {p2_prob:.1%}")
print("=" * 50)

### 5.1  Probability bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))
bars = ax.barh(
    [p2_name, p1_name],
    [p2_prob, p1_prob],
    color=["#DD8452", "#4C72B0"],
)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.axvline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.set_title(f"Win probability — {SURFACE.capitalize()}")
for bar, prob in zip(bars, [p2_prob, p1_prob]):
    ax.text(
        bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
        f"{prob:.1%}", va="center", fontsize=10,
    )
plt.tight_layout()
plt.show()

## 6  Surface comparison

Run the same match-up across all three surfaces.

In [ ]:
surfaces = ["hard", "clay", "grass"]
p1_probs = []

for surf in surfaces:
    xv = build_prediction_vector(
        p1_id, p2_id, surf,
        player_state, players_df, rankings, feature_names,
    )
    prob = model.predict_proba(xv.reshape(1, -1))[0]
    p1_probs.append(prob[classes.index("Player 1 Wins")])

fig, ax = plt.subplots(figsize=(5, 3))
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = ax.bar(surfaces, p1_probs, color=colors)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title(f"{p1_name} win probability by surface")
for bar, prob in zip(bars, p1_probs):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{prob:.1%}", ha="center", fontsize=10,
    )
plt.tight_layout()
plt.show()

## 7  ELO history for any player

Reconstruct a player's ELO trajectory from the grad-window state.

In [ ]:
PLOT_PLAYER = "Rafael Nadal"

pid, pname = find_player(PLOT_PLAYER, players_df)

# The longest ELO-grad window (250) stores the last 250 ELO values
elo_history = list(player_state["elo_grad"][250].get(pid, [ELO_DEFAULT]))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(elo_history, color="#4C72B0", linewidth=1.5)
ax.axhline(ELO_DEFAULT, color="grey", linestyle="--", linewidth=0.8, label="Starting ELO")
ax.set_title(f"{pname} — last {len(elo_history)} ELO values")
ax.set_xlabel("Match index (recent 250 max)")
ax.set_ylabel("ELO rating")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Current ELO: {player_state['elo'].get(pid, ELO_DEFAULT):.1f}")